## Future idea: Polars Lazy processing on the full Parquet dataset

Later, create a separate notebook for processing the full prepared dataset directly from Parquet files using Polars Lazy API.

Planned notebook name:

`10_polars_lazy_processing.ipynb`

Purpose:
- use `pl.scan_parquet()` instead of loading the full dataset eagerly,
- process the whole prepared NOAA dataset from Parquet files,
- filter observations for Polish weather stations,
- select relevant weather metrics,
- calculate yearly/monthly/station-level aggregates,
- collect only the final aggregated results into memory,
- compare the workflow conceptually with Pandas and regular Polars eager processing.

This notebook should demonstrate how Polars Lazy can optimize large data processing by delaying execution until `.collect()` and pushing filters/projections down before reading unnecessary data.

Full raw dataset contains over 422M observations and cannot be practically materialized on a 32 GB RAM laptop, so the pipeline uses Polars lazy execution, projection/filter pushdown, lazy joins, and aggregation before collection.

In [1]:
# Import libraries
import polars as pl
from pathlib import Path

# Path to the full Parquet dataset stored outside the project repository
WEATHER_PATH = Path(
    r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\weather.parquet"
)

STATIONS_PATH = Path(
    r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\stations.csv"
)

#### 1. Create LazyFrames and inspect schemas

In [2]:
lf_weather_raw = pl.scan_parquet(WEATHER_PATH)
lf_stations_raw = pl.scan_csv(STATIONS_PATH)

# Inspect schemas without loading the full dataset
# LazyFrames are not loaded into RAM yet
(lf_weather_raw.collect_schema(), lf_stations_raw.collect_schema())

(Schema([('station', String),
         ('observation_date', Int64),
         ('metric', String),
         ('value', Int64),
         ('measurement_flag', String),
         ('quality_flag', String),
         ('source_flag', String),
         ('observation_time', Float64)]),
 Schema([('station', String),
         ('latitude', Float64),
         ('longitude', Float64),
         ('elevation', Float64),
         ('state', String),
         ('station_name', String),
         ('gsn_flag', String),
         ('hcn_flag', String),
         ('wmo_id', Float64)]))

#### 2. Define transformations (column selection and filtering)
Select only the required columns and limit weather observations to the 2015–2025 analysis period (2026 data ends in June).

In [3]:
lf_weather = (
    lf_weather_raw
    .select(['station', 'observation_date', 'metric', 'value'])
    .filter(
        (pl.col('observation_date') >= 20150101) &
        (pl.col('observation_date') <= 20251231)
    )
)

lf_stations = (
    lf_stations_raw.select(
        ['station', 'station_name', 'elevation']
    ).with_columns(
        pl.col('station').str.slice(0, 2).alias('country_code') # Extract 2-letter country code from station ID
    )
)

#### 3. Count rows after filtering without loading the full dataset

In [4]:
(
    lf_weather.select(
        pl.len()
        .alias('weather_row_count'))
        .collect(), 
    lf_stations.select(
        pl.len()
        .alias('stations_row_count'))
        .collect()
)

(shape: (1, 1)
 ┌───────────────────┐
 │ weather_row_count │
 │ ---               │
 │ u32               │
 ╞═══════════════════╡
 │ 406304585         │
 └───────────────────┘,
 shape: (1, 1)
 ┌────────────────────┐
 │ stations_row_count │
 │ ---                │
 │ u32                │
 ╞════════════════════╡
 │ 132501             │
 └────────────────────┘)

#### 4. Filter the station metadata to include only European weather stations



In [5]:
# Source for mapping country codes to geographic regions
COUNTRIES = (
    "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv"
)

df_countries_raw = pl.read_csv(COUNTRIES)

df_countries_raw.head()

name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
str,str,str,i64,str,str,str,str,str,str,str
"""Afghanistan""","""AF""","""AFG""",4,"""ISO 3166-2:AF""","""Asia""","""Southern Asia""","""""","""142""","""034""",""""""
"""Åland Islands""","""AX""","""ALA""",248,"""ISO 3166-2:AX""","""Europe""","""Northern Europe""","""""","""150""","""154""",""""""
"""Albania""","""AL""","""ALB""",8,"""ISO 3166-2:AL""","""Europe""","""Southern Europe""","""""","""150""","""039""",""""""
"""Algeria""","""DZ""","""DZA""",12,"""ISO 3166-2:DZ""","""Africa""","""Northern Africa""","""""","""002""","""015""",""""""
"""American Samoa""","""AS""","""ASM""",16,"""ISO 3166-2:AS""","""Oceania""","""Polynesia""","""""","""009""","""061""",""""""


In [6]:
# Select country name, ISO alpha-2 code (country_code), and geographic region...
df_countries = df_countries_raw.select(pl.col(
    (['name', 'alpha-2', 'region'])
)).rename( # ...and rename alpha-2
    {'alpha-2':'country_code'}
)

# Convert the country lookup DataFrame to a LazyFrame for use in the lazy pipeline
lf_countries = df_countries.lazy()
lf_countries.collect_schema()

Schema([('name', String), ('country_code', String), ('region', String)])

In [7]:
# Join station metadata with country regions and keep only European stations
lf_eu_stations = lf_stations.join(
    lf_countries,
    on='country_code',
    how='left'
).filter(
    pl.col('region') == 'Europe'
)

# Inspect the resulting schema
lf_eu_stations.collect_schema()

Schema([('station', String),
        ('station_name', String),
        ('elevation', Float64),
        ('country_code', String),
        ('name', String),
        ('region', String)])

In [8]:
# Count European stations after filtering
lf_eu_stations.select(
    pl.len().alias('stations_row_count')
).collect()

stations_row_count
u32
3648


#### 5. Join the weather observations with station metadata using the station identifier


In [9]:
# Inspect column names before join
{
    'lf_eu_stations': lf_eu_stations.collect_schema().names(), 
    'lf_weather': lf_weather.collect_schema().names()
 }

{'lf_eu_stations': ['station',
  'station_name',
  'elevation',
  'country_code',
  'name',
  'region'],
 'lf_weather': ['station', 'observation_date', 'metric', 'value']}

In [10]:
lf_weather_stations = lf_weather.join(
    lf_eu_stations,
    on='station',
    how='inner'
)

lf_weather_stations

The optimized query plan shows that Polars applies two important optimizations before the join:

- **Projection pushdown** – Polars reads only the columns that are actually needed.
  - 4 of 8 columns are read from the weather Parquet file.
  - 3 of 9 columns are read from the station metadata CSV file.

- **Predicate pushdown** – Polars applies filters as early as possible during data scanning.
  - The 2015–2025 date filter is applied while scanning the Parquet file.

The left join is performed after these reductions.

As a result, Polars reads less data from disk and avoids loading the full raw dataset into memory before further processing.

#### 6. Keep only the weather metrics required for further processing and aggregation

In [ ]:
-- Example SQL query used to inspect the top 10 weather metrics
SELECT 
    metric,
    COUNT(*) AS metric_count 
FROM bronze.weather
GROUP BY metric
ORDER BY metric_count DESC
LIMIT 10;


#### The result of SQL query
| metric | metric_count |
|--------|-------------:|
| PRCP   | 124,482,318 |
| SNOW   | 56,183,306 |
| TMIN   | 51,404,903 |
| TMAX   | 51,344,828 |
| SNWD   | 37,043,127 |
| TAVG   | 24,557,801 |
| TOBS   | 19,429,221 |
| WESD   | 6,167,044 |
| AWND   | 4,656,731 |
| WSF2   | 4,402,701 |

#### Selected weather metrics

The analysis focuses on:

- **PRCP** – precipitation
- **SNWD** – snow depth

`SNOW` is excluded because it represents daily snowfall, while `SNWD` better reflects the actual snow conditions present on the ground.

Temperature metrics such as `TAVG`, `TMIN`, and `TMAX` are excluded because temperature was already analyzed in the previous Pandas and Polars notebooks.

In [18]:
# Inspect lf_weather_stations
lf_weather_stations.collect_schema().names()

['station',
 'observation_date',
 'metric',
 'value',
 'station_name',
 'elevation',
 'country_code',
 'name',
 'region']

In [ ]:
# Define metrics to include
metrics = [
    'PRCP', 'SNWD'
]

# Filter the weather_stations LazyFrame to selected metrics
lf_weather_stations = lf_weather_stations.filter(
    pl.col('metric').is_in(metrics)
)

lf_weather_stations.select(
    pl.col('metric').unique()
).collect()

metric
str
"""SNWD"""
"""PRCP"""


#### 7. Aggregate the Dataset

Aggregate the filtered data to reduce the full dataset to a smaller result, for example yearly statistics by station and metric.


#### 8. Inspect the Optimized Query Plan

Inspect the optimized lazy query plan before executing the transformations.


#### 9. Collect the Aggregated Result

Execute the lazy query and materialize only the aggregated result as a DataFrame.

In [ ]:
# df_weather = lf_processed_weather.collect()
# df_stations = lf_processed_stations.collect()

#### 10. Measure Execution Time

Measure the execution time of the full lazy processing pipeline.